In [122]:
import numpy as np
import cupy as cp
import pyuv_helper as ph
import importlib as il
il.reload(ph)

<module 'pyuv_helper' from '/home/thomasb/albatros_analysis/scripts/xcorr/pyuv_helper.py'>

In [123]:
nants = 4
npols = 2
nchans = 40
nantpol = nants*npols

In [124]:
#testing antenna indices to basline indices.
#first component of getting the rearrangement right
for i in range(nants):
    for j in range(i, nants):
        bl_idx = i * nants - (i * (i - 1)) // 2 + (j - i)
        print(f'({i, j}) to {bl_idx}')

((0, 0)) to 0
((0, 1)) to 1
((0, 2)) to 2
((0, 3)) to 3
((1, 1)) to 4
((1, 2)) to 5
((1, 3)) to 6
((2, 2)) to 7
((2, 3)) to 8
((3, 3)) to 9


In [125]:
#testing antpol indices into baseline and total pol indices
for i in range(nantpol):
    for j in range(i, nantpol):
        bline_idx, pol_idx = ph.antpol_to_bl(i, j, nants, npols)
        print(f'({i, j}) to {pol_idx, bline_idx}')

((0, 0)) to (0, 0)
((0, 1)) to (2, 0)
((0, 2)) to (0, 1)
((0, 3)) to (2, 1)
((0, 4)) to (0, 2)
((0, 5)) to (2, 2)
((0, 6)) to (0, 3)
((0, 7)) to (2, 3)
((1, 1)) to (1, 0)
((1, 2)) to (3, 1)
((1, 3)) to (1, 1)
((1, 4)) to (3, 2)
((1, 5)) to (1, 2)
((1, 6)) to (3, 3)
((1, 7)) to (1, 3)
((2, 2)) to (0, 4)
((2, 3)) to (2, 4)
((2, 4)) to (0, 5)
((2, 5)) to (2, 5)
((2, 6)) to (0, 6)
((2, 7)) to (2, 6)
((3, 3)) to (1, 4)
((3, 4)) to (3, 5)
((3, 5)) to (1, 5)
((3, 6)) to (3, 6)
((3, 7)) to (1, 6)
((4, 4)) to (0, 7)
((4, 5)) to (2, 7)
((4, 6)) to (0, 8)
((4, 7)) to (2, 8)
((5, 5)) to (1, 7)
((5, 6)) to (3, 8)
((5, 7)) to (1, 8)
((6, 6)) to (0, 9)
((6, 7)) to (2, 9)
((7, 7)) to (1, 9)


In [126]:
#set up index maps
bline_idx_map, pol_idx_map = ph.get_bl_pol_maps(nants, npols)
print('bline_idx map:')
print(bline_idx_map)
print('pol idx map')
print(pol_idx_map)

BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!
BEWARE: passing antenna in wrong order. flipping!


In [127]:
#testing mappping vs manual looping computation

counter = 0
sk = 1
for i in range(nantpol):
    sk ^= 1
    for j in range(i-sk, nantpol):
        bl_idx_manual, pol_tot_idx_manual = ph.antpol_to_bl(i, j, nants, npols)
        bl_idx_vector, pol_tot_idx_vector = bline_idx_map[i,j], pol_idx_map[i,j]
        assert bl_idx_manual == bl_idx_vector
        assert pol_tot_idx_manual == pol_tot_idx_vector
        print(f'{i,j} --> {bl_idx_manual, pol_tot_idx_manual}')
        counter += 1
assert counter == (nants*(nants+1)/2)*4

(0, 0) --> (0, 0)
(0, 1) --> (0, 2)
(0, 2) --> (1, 0)
(0, 3) --> (1, 2)
(0, 4) --> (2, 0)
(0, 5) --> (2, 2)
(0, 6) --> (3, 0)
(0, 7) --> (3, 2)
(1, 0) --> (0, 3)
(1, 1) --> (0, 1)
(1, 2) --> (1, 3)
(1, 3) --> (1, 1)
(1, 4) --> (2, 3)
(1, 5) --> (2, 1)
(1, 6) --> (3, 3)
(1, 7) --> (3, 1)
(2, 2) --> (4, 0)
(2, 3) --> (4, 2)
(2, 4) --> (5, 0)
(2, 5) --> (5, 2)
(2, 6) --> (6, 0)
(2, 7) --> (6, 2)
(3, 2) --> (4, 3)
(3, 3) --> (4, 1)
(3, 4) --> (5, 3)
(3, 5) --> (5, 1)
(3, 6) --> (6, 3)
(3, 7) --> (6, 1)
(4, 4) --> (7, 0)
(4, 5) --> (7, 2)
(4, 6) --> (8, 0)
(4, 7) --> (8, 2)
(5, 4) --> (7, 3)
(5, 5) --> (7, 1)
(5, 6) --> (8, 3)
(5, 7) --> (8, 1)
(6, 6) --> (9, 0)
(6, 7) --> (9, 2)
(7, 6) --> (9, 3)
(7, 7) --> (9, 1)


In [ ]:
#TESTING MANUAL AND VECTORIZED REFORMULATION
#to speed up refomulation while maintaining the (known to be true) manual assignment

#set up variables
row_shape = (nantpol, nantpol, nchans)
nbls = int(nants*(nants+1)/2)
npols_tot = 4
#set up data arrays
row_shape = (nantpol, nantpol, nchans)
row_new_shape = (nbls, nchans, npols_tot)
row_data = np.zeros(row_shape)
row_manual = np.zeros(row_new_shape)
row_vector = np.zeros(row_new_shape)

#generate data that is symmetric for each channel
for chan_idx in range(nchans):
    d = np.random.rand(nantpol, nantpol)
    d = np.triu(d)               
    chan_data = np.round(d + np.triu(d, 1).T, 3)
    row_data[:, :, chan_idx] = chan_data

#make arrays for data to be dumped into
row_manual, row_vector = np.zeros(row_new_shape), np.zeros(row_new_shape)

#manual assignment
sk = 1
for chan_idx in range(nchans):
    counter = 0
    for i in range(nantpol):
        sk ^= 1
        for j in range(i-sk, nantpol):
            bl_idx_manual, pol_tot_idx_manual = ph.antpol_to_bl(i, j, nants, npols)
            row_manual[bl_idx_manual, chan_idx, pol_tot_idx_manual] = row_data[i, j, chan_idx]
            #print(f'{i,j} --> {bl_idx_manual, pol_tot_idx_manual}')
            counter += 1
    assert counter == (nants*(nants+1)/2)*4 #ensure no overwrites


#vectorized assignment
#be careful here: need to transpose the pol_idx
#because vectorization will overwrite the lower triangular
#if we transpose, it maintains the correct alignment

for chan_idx in range(nchans):
    row_vector[bline_idx_map.ravel(), chan_idx, (pol_idx_map.T).ravel()] = row_data[:, :, chan_idx].ravel()


are_equal = np.array_equal(row_vector, row_manual)
print(are_equal)

#print(row_vector[:,0,:] - row_manual[:,0,:])
#print(row_manual[:,0,:])
#print(row_vector[:,0,:])

True


In [121]:
row_shape = (nantpol, nantpol)
row_new_shape = (nbls, 4)
print('row shape', row_shape)
print('new row shape', row_new_shape)


row shape (8, 8)
new row shape (10, 4)


In [138]:
np.arange(672*28, 687*28)
print(687*28 + 28)

19264
